#数据预处理

In [4]:
import os
import pandas as pd
from torchgen.packaged.autograd.gen_inplace_or_view_type import key

# 1. 在当前项目根目录下创建 data 文件夹
os.makedirs('data', exist_ok=True)

# 2. 文件路径（就在当前项目里）
data_file = os.path.join('data', 'house_tiny.csv')

# 3. 写入数据（所有 write 都在 with 里面）
with open(data_file, 'w') as f:
    f.write('NumRooms,Alley,Price\n')
    f.write('NA,Pave,127500\n')
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')

# 4. 验证是否真的创建成功
print("文件创建成功！完整路径：")
print(os.path.abspath(data_file))
print("文件是否存在：", os.path.exists(data_file))

# 5. 用 pandas 读取看看
data = pd.read_csv(data_file)
print(data)

文件创建成功！完整路径：
E:\STUDY_R\pytorch-learn\d2l_learn\day1\data\house_tiny.csv
文件是否存在： True
   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000


##处理缺失值

“NaN”项代表缺失值。为了处理缺失的数据,典型的方法包括插值法和删除法,其中插值法用一个替代值弥补缺失值,而删除法则直接忽略缺失值

In [9]:
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]
print("原始 inputs：")
print(inputs)
##inputs = inputs.fillna(inputs.mean())         试图对所有列求均值，但 Alley 是字符串类型（str / object），无法求均值，导致报错

# ==================== 关键修复 ====================
# 方法1（推荐，最清晰）
inputs = inputs.fillna(inputs.mean(numeric_only=True))

# 方法2（只对数值列填充，效果完全一样）
# numeric_cols = inputs.select_dtypes(include='number').columns
# inputs[numeric_cols] = inputs[numeric_cols].fillna(inputs[numeric_cols].mean())

print("\n填充缺失值后的 inputs：")
print(inputs)


原始 inputs：
   NumRooms Alley
0       NaN  Pave
1       2.0   NaN
2       4.0   NaN
3       NaN   NaN

填充缺失值后的 inputs：
   NumRooms Alley
0       3.0  Pave
1       2.0   NaN
2       4.0   NaN
3       3.0   NaN


pandas 的 pd.get_dummies()把类别型变量（离散值） 变成 0/1 数值 的过程，核心就是独热编码（One-Hot Encoding）

机器学习模型（比如后面的线性回归、神经网络）只能吃数字，不能直接吃 “Pave” 或 “NaN” 这种文字。
所以要把 “Alley” 这一列文字 → 拆成几列 0/1 数字

In [11]:
inputs = pd.get_dummies(inputs, dummy_na=True).astype(int)   #后面加上 .astype(int) 强制转为整数
print(inputs)

   NumRooms  Alley_Pave  Alley_nan
0         3           1          0
1         2           0          1
2         4           0          1
3         3           0          1


In [12]:
import torch
X = torch.tensor(inputs.to_numpy(dtype=float))
y = torch.tensor(outputs.to_numpy(dtype=float))
X, y

(tensor([[3., 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

pd.DataFrame(data) 专门支持下面这种字典格式：
data = {
    '列名1': [值1, 值2, 值3, ...],     # ← 这一列的所有数据
    '列名2': [值1, 值2, 值3, ...],
    '列名3': [值1, 值2, 值3, ...],
    ...
}

pandas 会自动做以下几件事：
把字典的 key 当成列名（NumRooms、Alley、HouseStyle...）
把字典的 value（必须是列表）当成这一列的所有数据
列表的长度决定总共有多少行（你写了10 个元素 → 10行）
所有列表长度必须一样（否则会报错）
None 会被自动转为 NaN（缺失值）

In [13]:
import os
import pandas as pd

# 1. 在当前项目里创建 data 文件夹
os.makedirs('data', exist_ok=True)

# 2. 新数据集文件路径
data_file = os.path.join('data', 'house_extended.csv')

# 3. 创建数据
data = {
    'NumRooms':    [3, 2, None, 4, None, 5, 3, None, 4, 2],
    'Alley':       ['Pave', None, 'Grvl', 'Pave', None, 'Pave', None, 'Grvl', 'Pave', None],
    'HouseStyle':  ['1Story', '2Story', '1Story', '2Story', '1.5Fin', '1Story', '2Story', '1Story', None, '2Story'],
    'YearBuilt':   [2000, 1995, 1980, None, 2010, 2005, None, 1970, 1998, 2008],
    'GarageCars':  [2, 1, 2, None, 3, 2, 1, 2, None, 1],
    'Price':       [300000, 180000, 250000, 320000, 400000, 280000, 350000, 220000, 310000, 260000]
}

df = pd.DataFrame(data)

# 4. 写入 CSV 文件
df.to_csv(data_file, index=False)

print("扩展数据集创建成功！")
print(f"文件路径：{os.path.abspath(data_file)}")
print(f"形状：{df.shape}（10 行 × 6 列）")
print("\n数据集预览：")
print(df)

扩展数据集创建成功！
文件路径：E:\STUDY_R\pytorch-learn\d2l_learn\day1\data\house_extended.csv
形状：(10, 6)（10 行 × 6 列）

数据集预览：
   NumRooms Alley HouseStyle  YearBuilt  GarageCars   Price
0       3.0  Pave     1Story     2000.0         2.0  300000
1       2.0  None     2Story     1995.0         1.0  180000
2       NaN  Grvl     1Story     1980.0         2.0  250000
3       4.0  Pave     2Story        NaN         NaN  320000
4       NaN  None     1.5Fin     2010.0         3.0  400000
5       5.0  Pave     1Story     2005.0         2.0  280000
6       3.0  None     2Story        NaN         1.0  350000
7       NaN  Grvl     1Story     1970.0         2.0  220000
8       4.0  Pave       None     1998.0         NaN  310000
9       2.0  None     2Story     2008.0         1.0  260000


In [24]:
data = pd.read_csv('data/house_extended.csv')   # ← 改成你当前的文件名

# ==================== 统计缺失值 ====================
missing_count = data.isnull().sum()
print(missing_count)
print(type(missing_count))
max_missing_col = data.isnull().sum().idxmax()  #返回列名
print(max_missing_col)
# 删除这一列
data_clean = data.drop(columns=[max_missing_col])
print(data_clean)

NumRooms      3
Alley         4
HouseStyle    1
YearBuilt     2
GarageCars    2
Price         0
dtype: int64
<class 'pandas.core.series.Series'>
Alley
   NumRooms HouseStyle  YearBuilt  GarageCars   Price
0       3.0     1Story     2000.0         2.0  300000
1       2.0     2Story     1995.0         1.0  180000
2       NaN     1Story     1980.0         2.0  250000
3       4.0     2Story        NaN         NaN  320000
4       NaN     1.5Fin     2010.0         3.0  400000
5       5.0     1Story     2005.0         2.0  280000
6       3.0     2Story        NaN         1.0  350000
7       NaN     1Story     1970.0         2.0  220000
8       4.0        NaN     1998.0         NaN  310000
9       2.0     2Story     2008.0         1.0  260000


In [26]:
inputs, outputs = data_clean.iloc[:, 0:4], data.iloc[:, 4]
print("原始 inputs：")
print(inputs)
##inputs = inputs.fillna(inputs.mean())         试图对所有列求均值，但 Alley 是字符串类型（str / object），无法求均值，导致报错


inputs = inputs.fillna(inputs.mean(numeric_only=True))

print("\n填充缺失值后的 inputs：")
print(inputs)



原始 inputs：
   NumRooms HouseStyle  YearBuilt  GarageCars
0       3.0     1Story     2000.0         2.0
1       2.0     2Story     1995.0         1.0
2       NaN     1Story     1980.0         2.0
3       4.0     2Story        NaN         NaN
4       NaN     1.5Fin     2010.0         3.0
5       5.0     1Story     2005.0         2.0
6       3.0     2Story        NaN         1.0
7       NaN     1Story     1970.0         2.0
8       4.0        NaN     1998.0         NaN
9       2.0     2Story     2008.0         1.0

填充缺失值后的 inputs：
   NumRooms HouseStyle  YearBuilt  GarageCars
0  3.000000     1Story    2000.00        2.00
1  2.000000     2Story    1995.00        1.00
2  3.285714     1Story    1980.00        2.00
3  4.000000     2Story    1995.75        1.75
4  3.285714     1.5Fin    2010.00        3.00
5  5.000000     1Story    2005.00        2.00
6  3.000000     2Story    1995.75        1.00
7  3.285714     1Story    1970.00        2.00
8  4.000000        NaN    1998.00        1.75
9  2.0

In [29]:
inputs = pd.get_dummies(inputs, dummy_na=True).astype(int)   #后面加上 .astype(int) 强制转为整数
print(inputs)

   NumRooms  YearBuilt  GarageCars  HouseStyle_1.5Fin  HouseStyle_1Story  \
0         3       2000           2                  0                  1   
1         2       1995           1                  0                  0   
2         3       1980           2                  0                  1   
3         4       1995           1                  0                  0   
4         3       2010           3                  1                  0   
5         5       2005           2                  0                  1   
6         3       1995           1                  0                  0   
7         3       1970           2                  0                  1   
8         4       1998           1                  0                  0   
9         2       2008           1                  0                  0   

   HouseStyle_2Story  HouseStyle_nan  
0                  0               0  
1                  1               0  
2                  0               0  
3      

In [30]:
import torch
X = torch.tensor(inputs.to_numpy(dtype=float))
y = torch.tensor(outputs.to_numpy(dtype=float))
X, y

(tensor([[3.0000e+00, 2.0000e+03, 2.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00,
          0.0000e+00],
         [2.0000e+00, 1.9950e+03, 1.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00,
          0.0000e+00],
         [3.0000e+00, 1.9800e+03, 2.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00,
          0.0000e+00],
         [4.0000e+00, 1.9950e+03, 1.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00,
          0.0000e+00],
         [3.0000e+00, 2.0100e+03, 3.0000e+00, 1.0000e+00, 0.0000e+00, 0.0000e+00,
          0.0000e+00],
         [5.0000e+00, 2.0050e+03, 2.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00,
          0.0000e+00],
         [3.0000e+00, 1.9950e+03, 1.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00,
          0.0000e+00],
         [3.0000e+00, 1.9700e+03, 2.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00,
          0.0000e+00],
         [4.0000e+00, 1.9980e+03, 1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
          1.0000e+00],
         [2.0000e+00, 2.0080e+03, 1.0000e+00, 0.0000e+0